# Hardware-Event Profiling vs Entropy for Packed Binary Detection

## 1. Setup

In [ ]:
!apt-get -qq install -y upx-ucl valgrind strace > /dev/null 2>&1
!pip -q install scikit-learn pandas matplotlib

## 2. Build Dataset (benign, UPX-packed, low-entropy packed)

In [ ]:
import os, re, shutil, subprocess, time
import numpy as np, pandas as pd

os.makedirs("bin", exist_ok=True)
names = "ls cat grep sort gzip sed wc head tail cut tr du base64 md5sum sha256sum uniq tee date env id basename dirname".split()

def run_ok(path):
    try:
        return subprocess.run([f"./{path}", "--version"], capture_output=True, timeout=20).returncode
    except Exception:
        return None

samples = []
for n in names:
    src = shutil.which(n)
    if not src: continue
    b, u, l = f"bin/{n}_benign", f"bin/{n}_upx", f"bin/{n}_upxpad"
    shutil.copy(src, b)
    subprocess.run(["upx", "-q", "--best", "-o", u, b], capture_output=True)
    if not os.path.exists(u): continue
    shutil.copy(u, l)
    with open(l, "ab") as f:                      # dilute whole-file entropy with zero padding
        f.write(b"\x00" * (8 * os.path.getsize(u)))
    ref = run_ok(b)
    if ref is None or run_ok(u) != ref or run_ok(l) != ref: continue
    samples += [(b, n, "benign", 0), (u, n, "upx", 1), (l, n, "upx_low_entropy", 1)]

df = pd.DataFrame(samples, columns=["path", "group", "kind", "label"])
df["kind"].value_counts()

## 3. Entropy Baseline

In [ ]:
def entropy(path):
    d = np.frombuffer(open(path, "rb").read(), dtype=np.uint8)
    p = np.bincount(d, minlength=256); p = p[p > 0] / len(d)
    return float(-(p * np.log2(p)).sum())

df["entropy"] = df.path.map(entropy)
df["pred_entropy"] = (df.entropy > 7.0).astype(int)
df.groupby("kind").entropy.describe()[["mean", "min", "max"]]

## 4. Hardware Event Profiling

In [ ]:
def prof_valgrind(path):
    r = subprocess.run(["valgrind", "--tool=cachegrind", "--branch-sim=yes", "--smc-check=all",
                        "--cachegrind-out-file=cg.out", f"./{path}", "--version"],
                       capture_output=True, text=True, timeout=300)
    keys = [r"I\s+refs", r"D\s+refs", r"D1\s+misses", r"LLd\s+misses", "Branches", "Mispredicts"]
    m = [re.search(rf"{k}\s*:\s*([\d,]+)", r.stderr) for k in keys]
    if not all(m):
        print("valgrind failed on", path, "|", r.stderr.strip().replace("\n", " ")[-200:]); return None
    ins, dr, d1, ll, br, mp = [int(x.group(1).replace(",", "")) for x in m]
    return dict(log_ins=np.log10(ins), br_per_ins=br / ins, mispred_rate=mp / max(br, 1),
                d1_miss_rate=d1 / max(dr, 1), ll_miss_per_kins=ll / ins * 1000)

def prof_strace(path):
    try:
        subprocess.run(["strace", "-f", "-o", "st.txt", f"./{path}", "--version"], capture_output=True, timeout=60)
        L = [l for l in open("st.txt") if "(" in l and "+++" not in l]
    except Exception as e:
        print("strace failed on", path, "|", e); return None
    if not L:
        print("strace produced no trace for", path); return None
    mm = [l for l in L if "mmap(" in l or "mprotect(" in l]
    return dict(log_syscalls=np.log10(len(L)),
                n_rwx=sum("PROT_WRITE" in l and "PROT_EXEC" in l for l in mm),
                n_anon_exec=sum("PROT_EXEC" in l and "MAP_ANONYMOUS" in l for l in mm),
                n_mprotect=sum("mprotect(" in l for l in L),
                n_self_exe=sum("/proc/self/exe" in l for l in L))

HELPER = ("import resource,subprocess,sys;subprocess.run([sys.argv[1],'--version'],capture_output=True);"
          "r=resource.getrusage(resource.RUSAGE_CHILDREN);print(r.ru_minflt,r.ru_maxrss,r.ru_utime+r.ru_stime)")

def prof_soft(path):
    o = subprocess.run(["python3", "-c", HELPER, f"./{path}"], capture_output=True, text=True).stdout.split()
    flt, rss, cpu = map(float, o)
    return dict(log_minflt=np.log10(flt + 1), log_maxrss=np.log10(rss + 1), log_cpu_ms=np.log10(cpu * 1000 + 1))

for backend, fn in [("valgrind cache/branch events", prof_valgrind), ("strace memory-permission events", prof_strace),
                    ("software events (page faults, RSS, CPU)", prof_soft)]:
    t, res = time.time(), []
    for p in df.path:
        r = fn(p)
        if r is None: break
        res.append(r)
    if len(res) == len(df): break
    print("backend unavailable, falling back\n")

print("Backend:", backend)
feature_cols = list(res[0])
df = pd.concat([df, pd.DataFrame(res)], axis=1)
df["prof_s"] = (time.time() - t) / len(df)
df.groupby("kind")[feature_cols].mean().round(4)

## 5. HPC-Based Detector (grouped cross-validation)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict

X = df[feature_cols]
clf = RandomForestClassifier(n_estimators=200, random_state=0)
df["pred_hpc"] = cross_val_predict(clf, X, df.label, groups=df.group, cv=GroupKFold(n_splits=5))

## 6. Comparison

In [ ]:
from sklearn.metrics import accuracy_score
res = pd.DataFrame({
    "detection_rate_upx": [df[df.kind == "upx"][c].mean() for c in ["pred_entropy", "pred_hpc"]],
    "detection_rate_low_entropy": [df[df.kind == "upx_low_entropy"][c].mean() for c in ["pred_entropy", "pred_hpc"]],
    "false_positive_rate": [df[df.kind == "benign"][c].mean() for c in ["pred_entropy", "pred_hpc"]],
    "overall_accuracy": [accuracy_score(df.label, df[c]) for c in ["pred_entropy", "pred_hpc"]],
}, index=["entropy (>7.0)", "HPC + random forest"])
print(res.round(3))
print(f"\nMean profiling time per binary: {df.prof_s.mean():.2f}s")

## 7. Visualization

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
col = {"benign": "tab:green", "upx": "tab:red", "upx_low_entropy": "tab:orange"}
fx, fy = feature_cols[1], feature_cols[0]
for k, g in df.groupby("kind"):
    ax[0].scatter(g.entropy, g[fy], c=col[k], label=k)
    ax[1].scatter(g[fx], g[fy], c=col[k], label=k)
ax[0].axvline(7.0, ls="--", c="gray"); ax[0].set(xlabel="File entropy", ylabel=fy, title="Entropy view")
ax[1].set(xlabel=fx, ylabel=fy, title="Event view"); ax[1].legend()
plt.tight_layout(); plt.show()